In [1]:
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


In [4]:
# ==========================================
# STEP 1: GENERATE SYNTHETIC DATA TO LANDING (FIXED)
# ==========================================
from pyspark.sql.functions import col, concat, lit, substring, rand, round, expr, monotonically_increasing_id

print("--- Generating Synthetic Data into Landing Zone ---")

# 1. Read customers directly from the landing zone CSV
df_customers = spark.read.csv("s3a://olist-data/landing/olist_customers_dataset.csv", header=True, inferSchema=True)
df_unique_customers = df_customers.select("customer_unique_id").dropDuplicates()

# 2. Generate CRM Data
df_crm_raw = df_unique_customers \
    .withColumn("email", concat(lit("user_"), substring(col("customer_unique_id"), 1, 8), lit("@gmail.com"))) \
    .withColumn("phone", concat(lit("+55-"), round(rand() * 90000 + 10000, 0).cast("int"), lit("-"), round(rand() * 9000 + 1000, 0).cast("int")))

# FIXED: Added .coalesce(1) to force a single CSV file output
df_crm_raw.coalesce(1).write.csv("s3a://olist-data/landing/crm_raw", mode="overwrite", header=True)
print("✅ CRM Raw CSV saved to Landing Zone.")

# 3. Generate Zendesk Data (Random 20% of customers)
df_zendesk_raw = df_crm_raw.sample(withReplacement=False, fraction=0.20, seed=42).select("email") \
    .withColumn("ticket_id", concat(lit("TKT-"), monotonically_increasing_id().cast("string"))) \
    .withColumn("issue_type", expr("CASE WHEN rand() < 0.5 THEN 'Late Delivery' ELSE 'Damaged Item' END")) \
    .withColumn("satisfaction_rating", expr("CAST(round(rand() * 4 + 1, 0) AS INT)"))

# FIXED: Added .coalesce(1) to force a single CSV file output
df_zendesk_raw.coalesce(1).write.csv("s3a://olist-data/landing/zendesk_raw", mode="overwrite", header=True)
print("✅ Zendesk Raw CSV saved to Landing Zone.")

--- Generating Synthetic Data into Landing Zone ---
✅ CRM Raw CSV saved to Landing Zone.
✅ Zendesk Raw CSV saved to Landing Zone.
